# Notebook 01: Data preparation and versioning

In this notebook, you create a tiny support-ticket dataset, add an instruction template, split it into train/eval sets, export JSONL files, and write a manifest.

The goal is not to train a real model. The goal is to practice the LLMOps habit: treat data and prompt structure as versioned artifacts.

In [1]:
from pathlib import Path
import datetime, hashlib, json
import pandas as pd

OUT = Path("artifacts_data_prep")
OUT.mkdir(exist_ok=True)

## 1. Create a small dataset

Each row has a customer message and an ideal reply.

In [2]:
examples = [
    {
        "ticket_id": "T-001",
        "customer_message": "My package arrived damaged and the box was open.",
        "ideal_reply": "I’m sorry the package arrived damaged. Please share a photo of the item and packaging, and I’ll help start a replacement or refund request.",
        "category": "delivery",
    },
    {
        "ticket_id": "T-002",
        "customer_message": "I was charged twice for my monthly plan.",
        "ideal_reply": "Thanks for flagging this. Please send the two charge dates or invoice numbers, and I’ll help check the duplicate billing issue.",
        "category": "billing",
    },
    {
        "ticket_id": "T-003",
        "customer_message": "The app keeps logging me out after the latest update.",
        "ideal_reply": "I’m sorry about the login trouble. Please confirm your app version and device type so we can narrow down what changed after the update.",
        "category": "technical",
    },
    {
        "ticket_id": "T-004",
        "customer_message": "Can I change the delivery address after placing the order?",
        "ideal_reply": "I can help check that. Please share your order number, and I’ll confirm whether the address can still be updated before shipment.",
        "category": "delivery",
    },
    {
        "ticket_id": "T-005",
        "customer_message": "I requested a refund last week but have not received an update.",
        "ideal_reply": "I understand the wait is frustrating. Please share your refund request number, and I’ll check the current status for you.",
        "category": "refund",
    },
]

df = pd.DataFrame(examples)
df

,ticket_id,customer_message,ideal_reply,category
0,T-001,My package arrived damaged and the box was open.,I’m sorry the package arrived damaged. Please ...,delivery
1,T-002,I was charged twice for my monthly plan.,Thanks for flagging this. Please send the two ...,billing
2,T-003,The app keeps logging me out after the latest ...,I’m sorry about the login trouble. Please conf...,technical
3,T-004,Can I change the delivery address after placin...,I can help check that. Please share your order...,delivery
4,T-005,I requested a refund last week but have not re...,I understand the wait is frustrating. Please s...,refund


## 2. Add an instruction template

The instruction is part of the input contract. If production later omits it, the model may see a different shape of input than the one used for evaluation or tuning.

In [3]:
INSTRUCTION_TEMPLATE = """You are a support assistant.
Draft a concise, polite reply that explains the next step.

Customer ticket:
"""

df["input_text"] = INSTRUCTION_TEMPLATE + df["customer_message"]
df["output_text"] = df["ideal_reply"]

df[["ticket_id", "input_text", "output_text"]].head(2)

,ticket_id,input_text,output_text
0,T-001,"You are a support assistant.\nDraft a concise,...",I’m sorry the package arrived damaged. Please ...
1,T-002,"You are a support assistant.\nDraft a concise,...",Thanks for flagging this. Please send the two ...


## 3. Split train/eval consistently

A fixed random seed makes the split reproducible. In real projects, changing the split changes the experiment, so record it.

In [4]:
SEED = 42
train = df.sample(frac=0.8, random_state=SEED)
eval_ = df.drop(train.index)

print("train rows:", len(train))
print("eval rows:", len(eval_))

train rows: 4
eval rows: 1


## 4. Export JSONL artifacts

Each line is one training/evaluation example. This is easy to inspect and works well for small beginner datasets.

In [6]:
date = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
train_path = OUT / f"support_reply_train_{date}.jsonl"
eval_path = OUT / f"support_reply_eval_{date}.jsonl"

cols = ["input_text", "output_text"]
train[cols].to_json(train_path, orient="records", lines=True, force_ascii=False)
eval_[cols].to_json(eval_path, orient="records", lines=True, force_ascii=False)

print(train_path)
print(eval_path)
print("First training line:")
print(train_path.read_text(encoding="utf-8").splitlines()[0])

artifacts_data_prep\support_reply_train_20260609_204310.jsonl
artifacts_data_prep\support_reply_eval_20260609_204310.jsonl
First training line:
{"input_text":"You are a support assistant.\nDraft a concise, polite reply that explains the next step.\n\nCustomer ticket:\nI was charged twice for my monthly plan.","output_text":"Thanks for flagging this. Please send the two charge dates or invoice numbers, and I’ll help check the duplicate billing issue."}


## 5. Write a manifest

The manifest gives future-you enough information to understand what produced the files.

In [7]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = {
    "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "source": "synthetic support-ticket examples for LLMOps course",
    "instruction_template_version": "support_reply_v1",
    "split_seed": SEED,
    "train_rows": len(train),
    "eval_rows": len(eval_),
    "artifacts": {
        "train_jsonl": str(train_path),
        "eval_jsonl": str(eval_path),
        "train_sha256": sha256_file(train_path),
        "eval_sha256": sha256_file(eval_path),
    },
}

manifest_path = OUT / f"manifest_{date}.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest_path)
print(manifest_path.read_text(encoding="utf-8"))

artifacts_data_prep\manifest_20260609_204310.json
{
  "created_at": "2026-06-09T20:43:18",
  "source": "synthetic support-ticket examples for LLMOps course",
  "instruction_template_version": "support_reply_v1",
  "split_seed": 42,
  "train_rows": 4,
  "eval_rows": 1,
  "artifacts": {
    "train_jsonl": "artifacts_data_prep\\support_reply_train_20260609_204310.jsonl",
    "eval_jsonl": "artifacts_data_prep\\support_reply_eval_20260609_204310.jsonl",
    "train_sha256": "c7ea5bdb0cb5ad6254c9af2d9edd1bdd7aeaaec3c76d0d29ec3da247c62d12bd",
    "eval_sha256": "a0e7f6de40e70bc5849f96fe5299e82f7ecb550e27156fc73a65554c87e12894"
  }
}


## Try it yourself

Change the instruction template so the assistant replies in a more formal enterprise support tone. Regenerate the JSONL files and manifest. Then compare the old and new manifest: what changed, and what stayed the same?